# ChuckleNet: Self-Contained Extraction + Training
## No Google Drive - Data from Kaggle directly

**This notebook:**
1. Downloads WavLM embeddings from Kaggle
2. Extracts prosody from Kaggle audio
3. Trains fusion model
4. Saves model to Kaggle output

**Runtime:** ~45-60 min on T4 GPU

In [ ]:
# @title Step 1: Install dependencies
!pip install -q kaggle librosa scikit-learn

In [ ]:
# @title Step 2: Download data from Kaggle
import os
import subprocess

# Kaggle credentials (use your own)
# Alternatively, use public datasets

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)
os.chdir(DATA_DIR)

# Download WavLM embeddings (public dataset)
!kaggle datasets download -d subhajitdas/chuckle-wavlm-555-videos -p {DATA_DIR} --unzip

# Download audio (public dataset) 
!kaggle datasets download -d subhajitdas/chuckle-vtt-audio-tar -p {DATA_DIR} --unzip

print('✅ Data downloaded')
!ls -la

In [ ]:
# @title Step 3: Load and verify data
import json
from pathlib import Path

WAVLM_DIR = Path(f'{DATA_DIR}/chuckle-wavlm-555-videos')

# Load WavLM data
wavlm_data = {}
for json_file in WAVLM_DIR.glob('*.json'):
    vid = json_file.stem
    with open(json_file) as f:
        data = json.load(f)
    wavlm_data[vid] = data['embeddings']

print(f'✅ Loaded {len(wavlm_data)} videos')
total = sum(len(v) for v in wavlm_data.values())
print(f'   Total utterances: {total}')

In [ ]:
# @title Step 4: Check audio availability
from pathlib import Path

AUDIO_DIR = Path(f'{DATA_DIR}/chuckle-vtt-audio-tar')

# Check available audio
audio_count = len(list(AUDIO_DIR.rglob('*.wav'))) + len(list(AUDIO_DIR.rglob('*.mp3'))) + len(list(AUDIO_DIR.rglob('*.m4a')))
print(f'Audio files found: {audio_count}')

# Create audio lookup
def get_audio_path(vid):
    for ext in ['.wav', '.mp3', '.m4a']:
        # Try direct
        p = AUDIO_DIR / f'{vid}{ext}'
        if p.exists():
            return str(p)
        # Try in subdirs
        for match in AUDIO_DIR.rglob(f'{vid}{ext}'):
            return str(match)
    return None

# Test
test_vids = list(wavlm_data.keys())[:3]
for vid in test_vids:
    path = get_audio_path(vid)
    print(f'{vid}: {"Found" if path else "Missing"}')

In [ ]:
# @title Step 5: Prosody extraction (21-dim)
import numpy as np
import librosa
import time

SR = 16000

def extract_prosody_21dim(y, sr):
    features = []
    
    # F0 (pitch) - 5 dims
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        features.extend([
            np.mean(f0_clean) if len(f0_clean) > 0 else 0,
            np.std(f0_clean) if len(f0_clean) > 0 else 0,
            np.max(f0_clean) if len(f0_clean) > 0 else 0,
            np.min(f0_clean) if len(f0_clean) > 0 else 0,
            np.sum(voiced_flag) / len(voiced_flag) if len(voiced_flag) > 0 else 0
        ])
    except:
        features.extend([0]*5)
    
    # Energy - 5 dims
    rms = librosa.feature.rms(y=y)[0]
    features.extend([
        np.mean(rms), np.std(rms), np.max(rms), np.min(rms),
        np.max(rms) - np.min(rms)
    ])
    
    # Duration - 2 dims
    features.extend([
        len(y) / sr,
        len(y) / sr / (np.sum(rms > np.mean(rms)) + 1)
    ])
    
    # Spectral - 5 dims
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    spec_flat = librosa.feature.spectral_flatness(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features.extend([
        np.mean(spec_cent), np.mean(spec_bw), np.mean(spec_flat),
        np.mean(zcr), np.std(zcr)
    ])
    
    # Voice quality - 4 dims
    try:
        hnr = librosa.effects.hpss(y)[1]
        hnr_val = np.mean(hnr) / (np.mean(np.abs(y)) + 1e-8)
    except:
        hnr_val = 0
    features.extend([hnr_val, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y))])
    
    return np.array(features, dtype=np.float32)

print('Extracting prosody for all videos...')
t0 = time.time()

prosody_data = {}
failed = []

for i, (vid, embeddings) in enumerate(wavlm_data.items()):
    audio_path = get_audio_path(vid)
    
    if not audio_path:
        failed.append(vid)
        prosody_data[vid] = [np.zeros(21, dtype=np.float32).tolist()] * len(embeddings)
        continue
    
    try:
        y, sr = librosa.load(audio_path, sr=SR, mono=True)
        if len(y.shape) > 1:
            y = y.mean(axis=1)
        
        video_prosody = []
        for emb in embeddings:
            start_sample = int(emb['start'] * SR)
            end_sample = int(emb['end'] * SR)
            if end_sample > len(y):
                end_sample = len(y)
            y_slice = y[start_sample:end_sample]
            
            if len(y_slice) < SR * 0.1:
                video_prosody.append(np.zeros(21, dtype=np.float32))
            else:
                prosody = extract_prosody_21dim(y_slice, SR)
                video_prosody.append(prosody)
        
        prosody_data[vid] = video_prosody
        
    except Exception as e:
        failed.append(vid)
        prosody_data[vid] = [np.zeros(21, dtype=np.float32).tolist()] * len(embeddings)
    
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (len(wavlm_data) - i - 1)
        print(f'{i+1}/{len(wavlm_data)} | ETA: {eta/60:.1f}min | Failed: {len(failed)}')

print(f'\n✅ Done! Prosody: {len(prosody_data)} | Failed: {len(failed)}')
print(f'Time: {(time.time()-t0)/60:.1f} min')

In [ ]:
# @title Step 6: Prepare training data
import numpy as np

# Flatten all data
all_emb = []
all_pros = []
all_labels = []

for vid, embeddings in wavlm_data.items():
    if vid not in prosody_data:
        continue
    for i, emb in enumerate(embeddings):
        all_emb.append(emb['embedding'])
        all_pros.append(prosody_data[vid][i])
        all_labels.append(emb.get('label', 0))

all_emb = np.array(all_emb, dtype=np.float32)
all_pros = np.array(all_pros, dtype=np.float32)
all_labels = np.array(all_labels, dtype=np.int64)

print(f'Total: {len(all_emb)} samples')
print(f'Positive: {sum(all_labels)} ({sum(all_labels)/len(all_labels)*100:.1f}%)')
print(f'WavLM dim: {all_emb.shape[1]}')
print(f'Prosody dim: {all_pros.shape[1]}')

In [ ]:
# @title Step 7: Train/Val/Test split + Training
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Split
X_train, X_test, pros_train, pros_test, y_train, y_test = train_test_split(
    all_emb, all_pros, all_labels, test_size=0.2, random_state=42, stratify=all_labels
)
X_train, X_val, pros_train, pros_val, y_train, y_val = train_test_split(
    X_train, pros_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

# Tensors
train_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(pros_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
val_ds = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(pros_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long)
)
test_ds = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(pros_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# @title Step 8: Model + Training (ALL FIXES APPLIED)
class FusionModel(nn.Module):
    def __init__(self, wavlm_dim=768, prosody_dim=21):
        super().__init__()
        self.prosody_proj = nn.Sequential(
            nn.Linear(prosody_dim, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(wavlm_dim + 32, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, 2)
        )
    
    def forward(self, wavlm_emb, prosody):
        prosody_feat = self.prosody_proj(prosody)
        x = torch.cat([wavlm_emb, prosody_feat], dim=-1)
        return self.classifier(x)

model = FusionModel().to(device)

# FIXED: Class weights [1.0, 2.5] + CrossEntropyLoss
class_weights = torch.tensor([1.0, 2.5], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS = 20
best_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    t0 = time.time()
    
    # Train
    model.train()
    train_loss = 0
    for emb_b, pros_b, labels_b in train_loader:
        emb_b = emb_b.to(device)
        pros_b = pros_b.to(device)
        labels_b = labels_b.to(device)
        
        optimizer.zero_grad()
        logits = model(emb_b, pros_b)
        loss = criterion(logits, labels_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # FIXED
        optimizer.step()
        train_loss += loss.item()
    
    scheduler.step()
    
    # Validate
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for emb_b, pros_b, labels_b in val_loader:
            emb_b = emb_b.to(device)
            pros_b = pros_b.to(device)
            logits = model(emb_b, pros_b)
            preds = torch.argmax(logits, dim=-1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels_b.numpy())
    
    val_f1 = f1_score(val_labels, val_preds, average='binary')
    epoch_time = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f} | Time: {epoch_time:.1f}s')
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = model.state_dict().copy()
        torch.save(best_state, '/content/best_model.pt')
        print(f'  ✅ New best!')

In [ ]:
# @title Step 9: Final Evaluation
model.load_state_dict(best_state)
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for emb_b, pros_b, labels_b in test_loader:
        emb_b = emb_b.to(device)
        pros_b = pros_b.to(device)
        logits = model(emb_b, pros_b)
        preds = torch.argmax(logits, dim=-1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels_b.numpy())

test_f1 = f1_score(test_labels, test_preds, average='binary')
print(f'\n🏆 Test F1: {test_f1:.4f}')
print(classification_report(test_labels, test_preds, target_names=['No Laughter', 'Laughter']))

# Save results
import json
results = {
    'test_f1': float(test_f1),
    'val_f1': float(best_f1),
    'n_train': len(train_ds),
    'n_val': len(val_ds),
    'n_test': len(test_ds),
    'n_failed_audio': len(failed)
}
with open('/content/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('\n✅ Results saved to /content/results.json')

# Next Steps
1. Download model from `/content/best_model.pt`
2. Or upload to Kaggle for persistent storage
3. Run autoreasearch loop for hyperparameter tuning